## Experimento 2: Sustracción del Promedio Global 

**Hipótesis:** Ya que el Experimento 1 demostró que eliminar frecuencias abruptamente (hacerlas cero) destruye los armónicos de la voz, intentaremos un enfoque aritmético más suave. Extraeremos el segmento inicial de ruido (0.5 segundos), calcularemos su magnitud promedio (un único valor escalar), y le restaremos ese promedio fijo a todo el arreglo unidimensional de la señal original.

In [5]:
import numpy as np
from scipy.io import wavfile
from scipy.fft import fft, ifft
from IPython.display import Audio

# 1. Leer y normalizar el archivo de audio original
fs, data = wavfile.read('Audio1.wav')
if len(data.shape) > 1:
    data = data[:, 0]
data = data / np.max(np.abs(data))

# 2. FFT global de todo el audio (Arreglo 1D)
senal_fft = fft(data)
magnitud_senal = np.abs(senal_fft)
fases_senal = np.angle(senal_fft)

In [ ]:
# 3. Extraer el segmento de ruido (Primeros 0.5 segundos)
ruido_segmento = data[:int(0.5 * fs)]
ruido_fft = fft(ruido_segmento)

# 4. Calcular el PROMEDIO del ruido (un solo número escalar)
promedio_ruido = np.mean(np.abs(ruido_fft))

# 5. Sustracción plana: Restar el promedio a todo el arreglo
# Multiplicamos por 15 para exagerar el efecto aritmético y que sea audible
magnitud_limpia = magnitud_senal - (promedio_ruido * 15.0)

# Forzar un piso para evitar magnitudes negativas
magnitud_limpia = np.maximum(magnitud_limpia, 0.001) 

# 6. Reconstruir la señal
senal_fft_exp2 = magnitud_limpia * np.exp(1j * fases_senal)
audio_exp2 = np.real(ifft(senal_fft_exp2))

# Exportar el audio fallido
wavfile.write('Audio2.wav', fs, audio_exp2.astype(np.float32))

NameError: name 'Audio2' is not defined

**Diagnóstico esperado:** Esta resta "plana" fracasará porque el ruido ambiental no es uniforme; contiene picos de energía. Al restar un valor promedio fijo a todo el arreglo, se atenuarán innecesariamente las frecuencias útiles de la voz humana (haciéndola sonar opaca o lejana), mientras que los picos más fuertes del ruido del ventilador sobrevivirán a la resta. Esto demostrará que el ruido no puede tratarse como una constante estática.